# Neural Fortran Examples: Self-Attention

Created with Grok Build

Source code by [Milan Curcic](https://github.com/milancurcic) and the neural-fortran contributors, from the [neural-fortran](https://github.com/modern-fortran/neural-fortran) repository. The library is released under the MIT license. Copyright (c) 2018-2025 neural-fortran contributors. The paper is Milan Curcic, [A parallel Fortran framework for neural networks and deep learning](https://doi.org/10.1145/3323057.3323059), *ACM SIGPLAN Fortran Forum* 38(1), 2019.

## Introduction

This is `example/mha_simple.f90`. It is the simple memorization loop with a self-attention layer in front of the dense output. One random input of shape $3 \times 8$ is drawn once and trained toward the same two-component target as `simple.f90`.

## The stack

```{literalinclude} Fortran_Code/Section_Examples_Mha/app/mha_simple.f90
:language: fortran
:lines: 1-24
```

`input(3, 8)` is a sequence of length 3 and a model width of 8. `self_attention(4)` asks for four heads. The head width has to divide the model width. The library sets

$$
d_k = \frac{8}{4} = 2
$$

and stops if the division is not exact. Each head computes scaled dot-product attention. With $Q$, $K$, and $V$ the slices of the query, key, and value for that head,

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{Q K^{\mathsf T}}{\sqrt{d_k}}\right) V.
$$

The scale in the source is $\sqrt{1/d_k}$, which is the same factor. The attention matrix for a head is $3 \times 3$.

`print_info` on this run reports 288 attention parameters. That count is three input projections and one output projection, each an $8 \times 8$ matrix plus a bias of length 8:

$$
3(8 \cdot 8 + 8) + (8 \cdot 8 + 8) = 288.
$$

`flatten` turns the $3 \times 8$ attention output into a vector of length 24. `dense(2)` then has $24 \cdot 2 + 2 = 50$ parameters and a sigmoid activation. The attention layer's activation field is blank in the print. The nonlinearity inside it is the softmax over the attention scores.


## The same training loop

```{literalinclude} Fortran_Code/Section_Examples_Mha/app/mha_simple.f90
:language: fortran
:lines: 26-36
```

`random_number` fills `x` once, before the loop, so every step sees the same input. The target is again $(0.123456,\ 0.246802)$. Each step is `forward`, `backward` with the default quadratic loss, and SGD with learning rate 1. The print is `predict(x)` every 50 steps, through step 500.


## Program

```{literalinclude} Fortran_Code/Section_Examples_Mha/app/mha_simple.f90
:language: fortran
```

The package file below builds this source with fpm. The Fortran program itself is the file from the repository.

```{literalinclude} Fortran_Code/Section_Examples_Mha/fpm.toml
:language: toml
```

## Build

The program above was compiled from the neural-fortran repository at commit `357e63d` (`Support of the argument stride for locally_connected2d_layer`) with fpm 0.10.1:

```text
fpm build --profile release
```

The compiler was gfortran 14.2.0. The log is that release build. It compiles the library and every program in `example/`, including this one. The `fpm.toml` on this page is a standalone package for the copied source. It takes neural-fortran as a git dependency, in the same form as the sine-wave page.


```{literalinclude} Fortran_Code/fpm_release_build.log
:language: text
```

## Run

The binary was `mha_simple` from the release build. Standard error was empty and the process exited 0. The input matrix is random, so both the attention parameters and the path belong to this run.

```{literalinclude} Fortran_Code/Section_Examples_Mha/mha_simple.out
:language: text
```

## What the run did

![Self-attention outputs](images/nf_mha_simple.png)

Step 0 prints 0.476483 and 0.484180, near the middle of the sigmoid range. At step 500 the outputs are 0.123610 and 0.246802. The second component matches the target at six decimals. The first is still $1.5 \times 10^{-4}$ high. The dense network on the simple page matched both components by step 450. Attention has more parameters, and it is not a faster way to memorize one constant.

## Conclusion

`self_attention(4)` plus `flatten` is enough to put a transformer block's attention into the same `forward` / `backward` / `update` loop as a dense network. The input shape fixes the sequence length and the model width, and the head count has to divide the width.